# RAG Pipeline with Advanced Retrieval Evaluation with RAGAS
# Generate Synthetic Data using RAGAS, build RAG Pipeline with  Advanced Retrievals, Evaluate Performance

"""
This notebook evaluates different retrieval strategies for building codes domain:
- Naive Retrieval (baseline)
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (Cohere Rerank)
- Ensemble Retrieval

Pipeline:
1. Connect to Qdrant Cloud vector database
2. Generate synthetic test data with RAGAS
3. Implement all retrieval strategies
4. Implement RAG with all retrival strategies
5. Evaluate performance with RAGAS metrics
6. Compare and analyze results
"""

# ============================================================================
# SECTION 1: DEPENDENCIES AND API CONFIGURATION
# ============================================================================

In [1]:
import os
import getpass
from uuid import uuid4
import copy
import time
import pandas as pd
import numpy as np
from datasets import Dataset  

# Set up API keys for external services
os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")
os.environ["QDRANT_API_KEY"] = getpass.getpass("🔐 Enter your Qdrant API Key: ")
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("Enter your Langchain API Key:")


In [2]:
# LangChain imports for vector store and embeddings
from qdrant_client import QdrantClient
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

# RAGAS imports for synthetic data generation and evaluation
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset.synthesizers import (
    default_query_distribution, 
    SingleHopSpecificQuerySynthesizer, 
    MultiHopAbstractQuerySynthesizer, 
    MultiHopSpecificQuerySynthesizer
)

# RAGAS metrics - CORRECTED
from ragas.metrics import (
    LLMContextRecall,
    ContextEntityRecall,
    LLMContextPrecisionWithReference,
    NonLLMContextPrecisionWithReference  # ← YOU WERE MISSING THIS
)
from ragas import RunConfig, EvaluationDataset, evaluate

# Advanced retrieval strategy imports
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.retrievers import ContextualCompressionRetriever, EnsembleRetriever
from langchain_cohere import CohereRerank

# RAG pipeline imports - YOU WERE MISSING THESE
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# Document loading imports - YOU WERE MISSING THESE  
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader

# LangChain tracing imports
from langchain.callbacks.tracers import LangChainTracer

# NLTK setup
import nltk
nltk.download('punkt', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

True

# ============================================================================
# SECTION 2: QDRANT CLOUD CONNECTION AND VECTOR STORE SETUP
# ============================================================================

In [3]:
print("Setting up Qdrant Cloud connection...")

# Initialize Qdrant client with cloud credentials
qdrant_client = QdrantClient(
    url="https://c95924f4-831b-407f-be42-8e424740487b.us-east-1-0.aws.cloud.qdrant.io",
    api_key=os.environ["QDRANT_API_KEY"],
)

# Initialize embedding model (same as used during document ingestion)
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

# Connect to vector store with proper field mapping for content retrieval
vectorstore = QdrantVectorStore(
    client=qdrant_client,
    collection_name="post_midterm_R",  # Collection name from Qdrant Cloud
    embedding=embedding_model,
    content_payload_key="content",  # Maps to your 'content' field in Qdrant
    metadata_payload_key="metadata"  # Optional: for additional metadata
)

print("✅ Connected to Qdrant Cloud vector store")

# Test connection and content retrieval
test_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
results = test_retriever.get_relevant_documents("What are the requirements for GFCI protection?")

print("🔍 Testing vector store connection:")
for doc in results:
    print(f"Content preview: {doc.page_content[:200]}...")
    print(f"Metadata: {doc.metadata}")
    print("-" * 50)

Setting up Qdrant Cloud connection...
✅ Connected to Qdrant Cloud vector store


/tmp/ipykernel_26432/1531599195.py:25: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  results = test_retriever.get_relevant_documents("What are the requirements for GFCI protection?")


🔍 Testing vector store connection:
Content preview: of an existing electrical system shall be required to meet
installation and equipment requirements in NFPA 99.
806.4 Residential occupancies. In Group R-2, R-3 and R-4
occupancies and buildings regula...
Metadata: {'_id': '9678ceb3-ea05-5e68-bc04-91cf260f0bee', '_collection_name': 'post_midterm_R'}
--------------------------------------------------
Content preview:  are intended exclusively to
improve the lateral force-resisting system and are not
required by other sections of this code shall not be required
to meet the requirements of Section 1609 or Section 16...
Metadata: {'_id': '184fa3e9-a880-503d-abd9-c7c76638fac6', '_collection_name': 'post_midterm_R'}
--------------------------------------------------
Content preview: 1.
Cables used for survivability of required critical
circuits shall be listed in accordance with UL 2196
and shall have a fire-resistance rating of not less
than 2 hours.
2.
Electrical circuit protec...
Metadata

# ============================================================================
# SECTION 3: SYNTHETIC DATA GENERATION WITH RAGAS
# ============================================================================

Load PDFs for RAGAS (vs feeding qdrant vector store) because
RAGAS works better with complete, coherent documents
Avoids fragmented chunks that might not make sense
Gives RAGAS more context to generate meaningful questions

In [8]:
print("Loading PDFs from local directory...")

# Load PDFs from local docs/data folder
from pathlib import Path
from collections import defaultdict
from langchain_core.documents import Document
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader

# Make randomness reproducible across runs (affects shuffles/sampling, not the LLM itself)
import os, random
import numpy as np

# Stabilize Python's hash-based operations
os.environ["PYTHONHASHSEED"] = "0"

# Seed Python's RNG and NumPy for deterministic sampling/shuffling
random.seed(42)
np.random.seed(42)

# Load raw documents
data_dir = Path("../docs/data")
raw_documents = DirectoryLoader(str(data_dir), glob="**/*.pdf", loader_cls=PyMuPDFLoader).load()

# Group pages by file for balanced sampling
by_file = defaultdict(list)
for d in raw_documents:
    by_file[d.metadata.get("source")].append(d)

print(f"Loaded {len(raw_documents)} raw pages from {len(by_file)} PDFs")

# Sample pages across PDFs for better diversity
sample_docs = []
pages_per_pdf = 5  # Adjust this number based on your needs

for src, pages in by_file.items():
    # Sort pages by page number for consistency
    pages = sorted(pages, key=lambda d: d.metadata.get("page", d.metadata.get("page_number", 0)))
    
    # Sample pages from this PDF (with replacement if needed)
    num_pages_to_sample = min(pages_per_pdf, len(pages))
    sampled_pages = random.sample(pages, num_pages_to_sample)
    sample_docs.extend(sampled_pages)

# Shuffle the final sample for good measure
random.shuffle(sample_docs)

print(f"Sampled {len(sample_docs)} pages across {len(by_file)} PDFs")
print(f"Using {len(sample_docs)} sampled pages for synthetic data generation")

# Set up RAGAS models with proper wrappers
generator_llm = LangchainLLMWrapper(ChatOpenAI(
    model="gpt-4o-mini",  # More capable model for reasoning tasks
    temperature=0.0,      # 0.0 = most deterministic responses (still subject to external factors)
    request_timeout=120   # Extended timeout for complex operations
))

generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

# Initialize synthetic data generator
generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)

# Define query distribution for different question types
query_distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),    # Simple factual questions
    (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),    # Complex reasoning questions
    (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),    # Multi-step specific questions
]

# Generate synthetic test dataset
try:
    print(f"Generating test dataset with {len(sample_docs)} documents...")
    dataset = generator.generate_with_langchain_docs(
        sample_docs,
        testset_size=10,
        query_distribution=query_distribution,
    )
    print("Dataset generation completed successfully!")
    print(f"Generated {len(dataset)} test samples")
        
    # Convert to pandas for inspection
    testset_df = dataset.to_pandas()
    print("Columns:", testset_df.columns.tolist())
    
except Exception as e:
    print(f"Error during dataset generation: {e}")
    print("Try reducing document count or testset_size further")
    testset_df = None
    dataset = None

# Display sample generated questions
if dataset and testset_df is not None:
    print("\n📋 Sample generated questions:")
    for idx, row in testset_df.head(3).iterrows():
        print(f"{idx + 1}. {row['user_input']}")
        print(f"   Synthesizer: {row['synthesizer_name']}")
        print("-" * 50)

# Final validation check
if dataset and len(dataset) > 0:
    print(f"✅ Ready for evaluation with {len(dataset)} questions")
else:
    print("❌ No dataset generated - check your PDF loading")

Loading PDFs from local directory...
Loaded 3545 raw pages from 4 PDFs
Sampled 20 pages across 4 PDFs
Using 20 sampled pages for synthetic data generation
Generating test dataset with 20 documents...


Applying HeadlinesExtractor:   0%|          | 0/13 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

...

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

Dataset generation completed successfully!
Generated 10 test samples
Columns: ['user_input', 'reference_contexts', 'reference', 'synthesizer_name']

📋 Sample generated questions:
1. What are the renewal requirements for licenses according to the NC General Statutes?
   Synthesizer: single_hop_specifc_query_synthesizer
--------------------------------------------------
2. Wht is Class II in fire safety?
   Synthesizer: single_hop_specifc_query_synthesizer
--------------------------------------------------
3. Can you tell me where portable fire extinguishers are needed in Group R-1 buildings and if there are any special rules for them?
   Synthesizer: single_hop_specifc_query_synthesizer
--------------------------------------------------
✅ Ready for evaluation with 10 questions


# ============================================================================
# SECTION 4: RAG PIPELINE AND ADVANCED RETRIEVAL STRATEGIES
# ============================================================================
"""
Build complete RAG (Retrieval-Augmented Generation) pipelines that combine:
1. Advanced retrieval strategies for finding relevant documents
2. Augmented prompting for structured responses  
3. LLM generation for final answers
"""

In [9]:
print("🔧 Setting up RAG pipelines and retrieval strategies...")

# RAG prompt template
RAG_PROMPT = """\
You are a North Carolina home inspection expert who answers questions based on provided context. 
You must only use the provided context, and cannot use your own knowledge. 
If the context doesn't contain the answer, respond with "I don't know".

### Question
{question}

### Context
{context}
"""

# Set up components
CHAT_MODEL = ChatOpenAI(model='gpt-4o-mini')
rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

def format_docs(docs):
    """Format list of documents into single context string"""
    return "\n\n".join([doc.page_content for doc in docs])

# 1. Naive Retriever (baseline)
naive_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

naive_rag_chain = (
    {
        "context": itemgetter("question") | naive_retriever | format_docs,
        "question": itemgetter("question")
    }
    | rag_prompt | CHAT_MODEL | StrOutputParser()
)

print("✅ Naive retriever ready")

# 2. BM25 Retriever
# CHANGED: Added conditional creation to handle cases where BM25 fails
# CHANGED: Better error messaging and graceful fallback to None
all_docs = vectorstore.similarity_search("", k=300)
valid_docs = [doc for doc in all_docs if doc.page_content.strip()]

if valid_docs:
    bm25_retriever = BM25Retriever.from_documents(valid_docs)
    bm25_retriever.k = 10
    
    bm25_rag_chain = (
        {
            "context": itemgetter("question") | bm25_retriever | format_docs,
            "question": itemgetter("question")
        }
        | rag_prompt | CHAT_MODEL | StrOutputParser()
    )
    print(f"✅ BM25 retriever created with {len(valid_docs)} documents")
else:
    print("❌ No valid documents for BM25")
    bm25_retriever = None

# 3. Multi-Query Retriever
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, 
    llm=ChatOpenAI(model="gpt-4o-mini")
)

multi_query_rag_chain = (
    {
        "context": itemgetter("question") | multi_query_retriever | format_docs,
        "question": itemgetter("question")
    }
    | rag_prompt | CHAT_MODEL | StrOutputParser()
)

print("✅ Multi-Query retriever created")

# 4. Contextual Compression with Cohere
compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, 
    base_retriever=naive_retriever
)

compression_rag_chain = (
    {
        "context": itemgetter("question") | compression_retriever | format_docs,
        "question": itemgetter("question")
    }
    | rag_prompt | CHAT_MODEL | StrOutputParser()
)

print("✅ Contextual Compression retriever created")

# 5. Ensemble Retriever
# CHANGED: Made conditional on BM25 success to prevent crashes
# CHANGED: Added warning message when skipped
if bm25_retriever:
    ensemble_retriever = EnsembleRetriever(
        retrievers=[naive_retriever, bm25_retriever],
        weights=[0.5, 0.5]
    )
    
    ensemble_rag_chain = (
        {
            "context": itemgetter("question") | ensemble_retriever | format_docs,
            "question": itemgetter("question")
        }
        | rag_prompt | CHAT_MODEL | StrOutputParser()
    )
    print("✅ Ensemble retriever created")
else:
    ensemble_retriever = None
    print("⚠️ Ensemble retriever skipped (no BM25)")

# Store retrievers for evaluation
# CHANGED: Dynamic retriever dictionary creation instead of hardcoded
# FIXED: Removed non-existent 'parent_document_retriever' reference
retrievers = {
    "Naive": naive_retriever,
    "MultiQuery": multi_query_retriever,
    "ContextualCompression": compression_retriever,
}

if bm25_retriever:
    retrievers["BM25"] = bm25_retriever
    
if ensemble_retriever:
    retrievers["Ensemble"] = ensemble_retriever

print(f"\n✅ Created {len(retrievers)} retrieval strategies")

# Test RAG pipeline
if dataset:
    test_question = testset_df.iloc[0]['user_input']
    test_answer = naive_rag_chain.invoke({"question": test_question})
    print(f"\n🧪 RAG Test - Q: {test_question}")
    print(f"A: {test_answer}")

🔧 Setting up RAG pipelines and retrieval strategies...
✅ Naive retriever ready
✅ BM25 retriever created with 300 documents
✅ Multi-Query retriever created
✅ Contextual Compression retriever created
✅ Ensemble retriever created

✅ Created 5 retrieval strategies

🧪 RAG Test - Q: What are the renewal requirements for licenses according to the NC General Statutes?
A: To renew an active home inspector license in North Carolina, the licensee must complete 12 credit hours of continuing education during the license period (October 1 through the following September 30). Additionally, the licensee must file an application for renewal with the NC Home Inspector Licensure Board, pay the required renewal fee, and not be in violation of the relevant statutes when the application is filed. The Board will notify license holders at least 30 days before their licenses expire. If the Board imposes continuing education requirements, it must ensure that the necessary courses are available in all geographic

# ============================================================================
# SECTION 5: RAG PIPELINE EVALUATION: Evaluation of EACH RETRIVAL against EACH CHUNKIN strategy for EACH of the 4 RAGAS metrics
# ============================================================================

In [18]:
# Evaluate each chunking strategy (experiment_label) separately
# - Restricts all retrieval to a single experiment_label via Qdrant filter
# - Builds Naive, MultiQuery, ContextualCompression, BM25, Ensemble per label

from qdrant_client.models import Filter, FieldCondition, MatchValue
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.retrievers import ContextualCompressionRetriever, EnsembleRetriever
from langchain_cohere import CohereRerank
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from operator import itemgetter

# Prompt + model
RAG_PROMPT = """\
You are a North Carolina home inspection expert who answers questions based on provided context. 
You must only use the provided context, and cannot use your own knowledge. 
If the context doesn't contain the answer, respond with "I don't know".

### Question
{question}

### Context
{context}
"""
CHAT_MODEL = ChatOpenAI(model="gpt-4o-mini")

def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

def get_unique_experiment_labels(collection_name="post_midterm_R", limit=None):
    labels = set()
    offset = None
    while True:
        batch, offset = qdrant_client.scroll(
            collection_name=collection_name,
            limit=1000,
            offset=offset,
            with_payload=["experiment_label"],
        )
        if not batch:
            break
        for pt in batch:
            lbl = (pt.payload or {}).get("experiment_label")
            if lbl:
                labels.add(lbl)
        if not offset or (limit and len(labels) >= limit):
            break
    return sorted(labels)

from langchain_core.documents import Document
def fetch_docs_for_label_from_qdrant(experiment_label: str, max_docs: int | None = 500): #fetch raw text documents for a single chunking strategy by  scrolling the entire Qdrant collection to ensure BM25 is built over the full corpus for that strategy, not an arbitrary similarity slice.
   
    docs = []
    offset = None
    fetched = 0
    while True:
        batch, offset = qdrant_client.scroll(
            collection_name="post_midterm_R",
            limit=1000,
            offset=offset,
            with_payload=["content", "experiment_label"],
        )
        if not batch:
            break
        for pt in batch:
            payload = pt.payload or {}
            if payload.get("experiment_label") == experiment_label:
                text = payload.get("content", "")
                if text and text.strip():
                    docs.append(Document(page_content=text, metadata={}))
                    fetched += 1
                    if max_docs and fetched >= max_docs:
                        return docs
        if not offset:
            break
    return docs

TOP_K = 10  # Number of contexts each retriever returns. See notes below on impact.

def build_filtered_retrievers_for_label(experiment_label, k: int = TOP_K):
    """
    Build per-label retrievers and RAG chains that are restricted to a single
    chunking strategy via a Qdrant filter. All retrievers use the same top-k to
    keep comparisons fair.
    """
    # Filter that restricts all vector queries to the given experiment_label
    flt = Filter(must=[FieldCondition(key="experiment_label", match=MatchValue(value=experiment_label))])

    # Naive retriever (vector search) restricted to experiment_label
    naive_retriever = vectorstore.as_retriever(search_kwargs={"k": k, "filter": flt})

    # Multi-Query retriever on top of the filtered naive retriever
    multi_query_retriever = MultiQueryRetriever.from_llm(
        retriever=naive_retriever,
        llm=ChatOpenAI(model="gpt-4o-mini")
    )

    # Contextual compression retriever (Cohere rerank) on top of filtered naive
    compressor = CohereRerank(model="rerank-v3.5")
    compression_retriever = ContextualCompressionRetriever(
        base_compressor=compressor,
        base_retriever=naive_retriever
    )

    # BM25 from docs restricted to this experiment_label
    valid_docs = fetch_docs_for_label_from_qdrant(experiment_label, max_docs=500)

    bm25_retriever = None
    if valid_docs:
        bm25_retriever = BM25Retriever.from_documents(valid_docs)
        bm25_retriever.k = k

    # Ensemble (only if BM25 exists) – both components restricted to label
    ensemble_retriever = None
    if bm25_retriever:
        ensemble_retriever = EnsembleRetriever(
            retrievers=[naive_retriever, bm25_retriever],
            weights=[0.5, 0.5]
        )

    retrievers = {
        "Naive": naive_retriever,
        "MultiQuery": multi_query_retriever,
        "ContextualCompression": compression_retriever,
    }
    if bm25_retriever:
        retrievers["BM25"] = bm25_retriever
    if ensemble_retriever:
        retrievers["Ensemble"] = ensemble_retriever

    # Build RAG chains for each retriever
    rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)
    rag_chains = {}
    for name, r in retrievers.items():
        rag_chains[name] = (
            {
                "context": itemgetter("question") | r | format_docs,
                "question": itemgetter("question"),
            }
            | rag_prompt | CHAT_MODEL | StrOutputParser()
        )

    return retrievers, rag_chains

if dataset is None or testset_df is None:
    print("❌ Cannot evaluate - synthetic data generation failed")
else:
    # Discover labels present in the collection
    labels_to_test = get_unique_experiment_labels("post_midterm_R")
    print("📦 Evaluating chunking strategies (experiment_label):")
    for lbl in labels_to_test:
        print(" -", lbl)

    # RAGAS eval setup
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
    os.environ["LANGCHAIN_PROJECT"] = f"BuildingCodes-RAGEval-{uuid4().hex[0:8]}"
    custom_run_config = RunConfig(timeout=360)
    evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini"))

    from ragas.metrics import LLMContextPrecisionWithReference, ResponseRelevancy, Faithfulness, LLMContextRecall

    all_results = {}  # {label: {rag_name: result}}

    for experiment_label in labels_to_test:
        print("\n" + "="*72)
        print(f"🔬 Evaluating CHUNKING STRATEGY: {experiment_label}")
        print("="*72)

        retrievers_for_label, rag_chains_for_label = build_filtered_retrievers_for_label(experiment_label, k=10)

        print(f"📊 Evaluating {len(rag_chains_for_label)} RAG pipelines for label '{experiment_label}'")

        label_results = {}
        for rag_name, rag_chain in rag_chains_for_label.items():
            print("\n" + "="*50)
            print(f"Evaluating {rag_name} (label: {experiment_label})")
            print("="*50)

            test_dataset_copy = copy.deepcopy(dataset)

            for test_row in test_dataset_copy:
                if rag_name in ["ContextualCompression", "Ensemble"]:
                    time.sleep(6.1)  # Cohere rate limit buffer
                try:
                    question = test_row.eval_sample.user_input
                    response = rag_chain.invoke({"question": question})
                    test_row.eval_sample.response = response

                    # Get retrieved contexts with the matching retriever
                    docs = retrievers_for_label[rag_name].invoke(question)
                    test_row.eval_sample.retrieved_contexts = [d.page_content for d in docs]
                except Exception as e:
                    print(f"⚠️ Error processing {rag_name}: {e}")
                    test_row.eval_sample.response = "Error generating response"
                    test_row.eval_sample.retrieved_contexts = []

            evaluation_dataset = EvaluationDataset.from_pandas(test_dataset_copy.to_pandas())
            tracer = LangChainTracer(project_name=f"{os.environ['LANGCHAIN_PROJECT']}-{experiment_label}-{rag_name}")

            try:
                result = evaluate(
                    dataset=evaluation_dataset,
                    metrics=[
                        LLMContextPrecisionWithReference(),
                        ResponseRelevancy(),
                        Faithfulness(),
                        LLMContextRecall(),
                    ],
                    llm=evaluator_llm,
                    run_config=custom_run_config,
                    callbacks=[tracer],
                )
                label_results[rag_name] = result
                # Inline summary
                try:
                    df = result.to_pandas()
                    precision = df["llm_context_precision_with_reference"].mean()
                    relevancy = df["answer_relevancy"].mean()
                    faith = df["faithfulness"].mean()
                    recall = df["context_recall"].mean()
                    overall = (precision + relevancy + faith + recall) / 4
                    print(f"📊 {rag_name} @ {experiment_label}: "
                          f"Prec={precision:.4f} Rel={relevancy:.4f} Faith={faith:.4f} Recall={recall:.4f} "
                          f"Overall={overall:.4f}")
                except Exception:
                    print(f"✅ {rag_name} evaluation done (summary table unavailable)")
            except Exception as e:
                print(f"❌ Error evaluating {rag_name} for {experiment_label}: {e}")

        all_results[experiment_label] = label_results

    print("\n✅ Completed evaluations per chunking strategy (experiment_label).")

📦 Evaluating chunking strategies (experiment_label):
 - heading_semantic_pack_600_80_hsp_600_80
 - recursive_1000_200_rec_1
 - token_512_64_tok_384_64

🔬 Evaluating CHUNKING STRATEGY: heading_semantic_pack_600_80_hsp_600_80
📊 Evaluating 5 RAG pipelines for label 'heading_semantic_pack_600_80_hsp_600_80'

Evaluating Naive (label: heading_semantic_pack_600_80_hsp_600_80)


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

📊 Naive @ heading_semantic_pack_600_80_hsp_600_80: Prec=0.7483 Rel=0.2831 Faith=0.5340 Recall=0.6833 Overall=0.5622

Evaluating MultiQuery (label: heading_semantic_pack_600_80_hsp_600_80)


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

📊 MultiQuery @ heading_semantic_pack_600_80_hsp_600_80: Prec=0.6980 Rel=0.4772 Faith=0.4938 Recall=0.7167 Overall=0.5964

Evaluating ContextualCompression (label: heading_semantic_pack_600_80_hsp_600_80)


...

Exception raised in Job[10]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-m71YBvwb05p6pFMRwdurLpCp on requests per day (RPD): Limit 10000, Used 10000, Requested 1. Please try again in 8.64s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'requests', 'param': None, 'code': 'rate_limit_exceeded'}})
Exception raised in Job[2]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-m71YBvwb05p6pFMRwdurLpCp on tokens per min (TPM): Limit 200000, Used 199495, Requested 6097. Please try again in 1.677s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}})
Exception raised in Job[8]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-m71YBvwb05p6pFMRwdurLpCp on requests per day (RPD): Limit 10000, Use

📊 BM25 @ token_512_64_tok_384_64: Prec=nan Rel=0.0957 Faith=0.0000 Recall=0.5000 Overall=nan

Evaluating Ensemble (label: token_512_64_tok_384_64)


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

Exception raised in Job[28]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-m71YBvwb05p6pFMRwdurLpCp on tokens per min (TPM): Limit 200000, Used 200000, Requested 1705. Please try again in 511ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}})
Exception raised in Job[4]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-m71YBvwb05p6pFMRwdurLpCp on tokens per min (TPM): Limit 200000, Used 200000, Requested 1681. Please try again in 504ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}})
Exception raised in Job[21]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-m71YBvwb05p6pFMRwdurLpCp on requests per day (RPD): Limit 10000, Us

📊 Ensemble @ token_512_64_tok_384_64: Prec=nan Rel=0.6002 Faith=0.8589 Recall=0.8438 Overall=nan

✅ Completed evaluations per chunking strategy (experiment_label).


# ============================================================================
# SECTION 6: RAG PIPELINE RESULTS ANALYSIS for EACH chunking strategy For EACH RAG pipeline For EACH of the 4 RAGAS metrics
# ============================================================================

In [ ]:
# Analysis of evaluation results (per-label), with NaN-safe handling and clear summaries

import numpy as np
import pandas as pd

# all_results is expected to be: {experiment_label: {rag_name: ragas_result}}
if 'all_results' not in globals() or not all_results:
    print("❌ No chunking strategy results to analyze - evaluation failed")
else:
    print("\n📊 Analyzing chunking strategy × RAG pipeline results...")

    metrics = ["Context Precision", "Response Relevancy", "Faithfulness", "Context Recall"]
    comparison_data = []

    # Flatten all_results into rows
    for experiment_label, label_results in all_results.items():
        for rag_name, result in label_results.items():
            if result is None:
                continue
            try:
                df = result.to_pandas()
                # Explicitly use skipna=True for clarity
                row = {
                    "Chunking Strategy": experiment_label,
                    "RAG Pipeline": rag_name,
                    "Context Precision": df["llm_context_precision_with_reference"].mean(skipna=True),
                    "Response Relevancy": df["answer_relevancy"].mean(skipna=True),
                    "Faithfulness": df["faithfulness"].mean(skipna=True),
                    "Context Recall": df["context_recall"].mean(skipna=True),
                }
            except Exception:
                # Fallback: try direct attributes if pandas conversion isn't available
                row = {
                    "Chunking Strategy": experiment_label,
                    "RAG Pipeline": rag_name,
                    "Context Precision": getattr(result, "llm_context_precision_with_reference", np.nan),
                    "Response Relevancy": getattr(result, "answer_relevancy", np.nan),
                    "Faithfulness": getattr(result, "faithfulness", np.nan),
                    "Context Recall": getattr(result, "context_recall", np.nan),
                }
            comparison_data.append(row)

    comparison_df = pd.DataFrame(comparison_data)

    if comparison_df.empty:
        print("❌ No results to analyze")
    else:
        # Coerce metric columns to numeric; non-numeric → NaN
        for m in metrics:
            comparison_df[m] = pd.to_numeric(comparison_df[m], errors="coerce")

        # Drop rows with all metrics missing (nothing to score)
        valid_mask = comparison_df[metrics].notna().any(axis=1)
        dropped_rows = len(comparison_df) - valid_mask.sum()
        if dropped_rows > 0:
            print(f"ℹ️ Dropping {dropped_rows} combinations with no metric data")
        comparison_df = comparison_df[valid_mask].copy()

        # Overall = mean of available metrics (row-wise), skipping NaNs
        comparison_df["Overall Score"] = comparison_df[metrics].mean(axis=1, skipna=True)

        # Display full matrix (use "-" for NaN for better readability)
        print(f"\n📋 COMPLETE RESULTS MATRIX ({len(comparison_df)} combinations):")
        print("=" * 120)
        display_df = comparison_df.round(4).fillna("-")
        print(display_df.to_string(index=False))

        # Save clean CSV with original NaNs preserved (analysts can choose imputation later)
        out_path = "chunking_rag_metrics_results.csv"
        comparison_df.round(6).to_csv(out_path, index=False)
        print(f"\n💾 Saved CSV: {out_path}")

        if comparison_df.empty:
            print("\n⚠️ No valid combinations to rank")
        else:
            # Top overall performers
            comparison_df_sorted = comparison_df.sort_values("Overall Score", ascending=False)
            print(f"\n🏆 TOP 10 OVERALL PERFORMERS:")
            for i, (_, r) in enumerate(comparison_df_sorted.head(10).iterrows(), start=1):
                print(f"{i:2d}. {r['Chunking Strategy']:<45} + {r['RAG Pipeline']:<20} = {r['Overall Score']:.4f}")

            # Averages by chunking strategy (across pipelines) with data point counts
            print("\n🧩 CHUNKING STRATEGY (averages across pipelines):")
            chunking_summary = comparison_df.groupby("Chunking Strategy")[metrics + ["Overall Score"]].mean().round(4)
            chunking_counts = comparison_df.groupby("Chunking Strategy")[["Overall Score"]].count()
            chunking_counts.columns = ["N"]
            chunking_combined = pd.concat([chunking_summary, chunking_counts], axis=1)
            print(chunking_combined.sort_values("Overall Score", ascending=False).fillna("-").to_string())

            # Averages by RAG pipeline (across chunking strategies) with data point counts
            print("\n🔗 RAG PIPELINE (averages across chunking strategies):")
            rag_summary = comparison_df.groupby("RAG Pipeline")[metrics + ["Overall Score"]].mean().round(4)
            rag_counts = comparison_df.groupby("RAG Pipeline")[["Overall Score"]].count()
            rag_counts.columns = ["N"]
            rag_combined = pd.concat([rag_summary, rag_counts], axis=1)
            print(rag_combined.sort_values("Overall Score", ascending=False).fillna("-").to_string())

            # Best per metric (combination) with NaN guard
            print("\n🎯 BEST PERFORMERS BY METRIC:")
            for m in metrics:
                col = comparison_df[m]
                valid_count = col.notna().sum()
                if col.notna().any():
                    best_idx = col.idxmax()
                    r = comparison_df.loc[best_idx]
                    print(f"🏆 {m}: {r['Chunking Strategy']} + {r['RAG Pipeline']} = {r[m]:.4f} (Overall {r['Overall Score']:.4f}) [{valid_count} valid]")
                else:
                    print(f"🏆 {m}: no data")

            # Per-metric pivot tables (strategy × pipeline). Use "-" for NaN for readability.
            print("\n📈 METRIC PIVOTS (rows=Chunking Strategy, cols=RAG Pipeline):")
            for m in metrics:
                print(f"\n{m}:")
                pivot = comparison_df.pivot(index="Chunking Strategy", columns="RAG Pipeline", values=m).round(4).fillna("-")
                print(pivot.to_string())
                
                # Show count of valid values
                valid_pivot = comparison_df.pivot(index="Chunking Strategy", columns="RAG Pipeline", values=m).notna().sum().sum()
                total_cells = len(comparison_df["Chunking Strategy"].unique()) * len(comparison_df["RAG Pipeline"].unique())
                print(f"  Valid data points: {valid_pivot}/{total_cells}")

            # Within each pipeline: rank strategies by Overall
            print("\n📊 CHUNKING STRATEGY RANKS BY RAG PIPELINE (by Overall):")
            for pipeline in comparison_df["RAG Pipeline"].unique():
                subset = comparison_df[comparison_df["RAG Pipeline"] == pipeline].sort_values("Overall Score", ascending=False)
                valid_count = subset["Overall Score"].notna().sum()
                print(f"\n{pipeline} ({valid_count} strategies with data):")
                for _, r in subset.iterrows():
                    if pd.notna(r['Overall Score']):
                        print(f"  {r['Overall Score']:.4f} - {r['Chunking Strategy']}")
                    else:
                        print(f"  N/A - {r['Chunking Strategy']}")

            # Within each strategy: rank pipelines by Overall
            print("\n📊 RAG PIPELINE RANKS BY CHUNKING STRATEGY (by Overall):")
            for strategy in comparison_df["Chunking Strategy"].unique():
                subset = comparison_df[comparison_df["Chunking Strategy"] == strategy].sort_values("Overall Score", ascending=False)
                valid_count = subset["Overall Score"].notna().sum()
                print(f"\n{strategy} ({valid_count} pipelines with data):")
                for _, r in subset.iterrows():
                    if pd.notna(r['Overall Score']):
                        print(f"  {r['Overall Score']:.4f} - {r['RAG Pipeline']}")
                    else:
                        print(f"  N/A - {r['RAG Pipeline']}")

            # Summary statistics
            print("\n📈 SUMMARY STATISTICS:")
            print(f"Total combinations evaluated: {len(comparison_df)}")
            for m in metrics:
                valid_count = comparison_df[m].notna().sum()
                if valid_count > 0:
                    mean_val = comparison_df[m].mean(skipna=True)
                    std_val = comparison_df[m].std(skipna=True)
                    min_val = comparison_df[m].min(skipna=True)
                    max_val = comparison_df[m].max(skipna=True)
                    print(f"{m}: μ={mean_val:.4f}, σ={std_val:.4f}, range=[{min_val:.4f}, {max_val:.4f}], n={valid_count}")
                else:
                    print(f"{m}: No valid data")

            print("\n🚀 DONE")


📊 Analyzing chunking strategy × RAG pipeline results...

📋 COMPLETE RESULTS MATRIX (15 combinations):
                      Chunking Strategy          RAG Pipeline Context Precision  Response Relevancy  Faithfulness  Context Recall  Overall Score
heading_semantic_pack_600_80_hsp_600_80                 Naive            0.7483              0.2831        0.5340          0.6833         0.5622
heading_semantic_pack_600_80_hsp_600_80            MultiQuery             0.698              0.4772        0.4938          0.7167         0.5964
heading_semantic_pack_600_80_hsp_600_80 ContextualCompression            0.8833              0.4719        0.5983          0.5667         0.6301
heading_semantic_pack_600_80_hsp_600_80                  BM25            0.3479              0.0971        0.0750          0.5250         0.2613
heading_semantic_pack_600_80_hsp_600_80              Ensemble            0.6089              0.3785        0.5359          0.7667         0.5725
               recursive_10

In [23]:
result = all_results["token_512_64_tok_384_64"]["BM25"]
print(result.to_pandas()["llm_context_precision_with_reference"])

0   NaN
1   NaN
2   NaN
3   NaN
4   NaN
5   NaN
6   NaN
7   NaN
8   NaN
9   NaN
Name: llm_context_precision_with_reference, dtype: float64


In [24]:
# Check if any contexts were retrieved
print(result.to_pandas()["contexts"])  # Should not be empty

KeyError: 'contexts'